# Lab 03 · Prompt chaining with quality gates

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · 15–20 minute lab

## Chain a small DDD design exercise
**So far:** Lab 02 retrieved lecture evidence.

**New:** prompt chaining with quality gates. Use ShopEasy's events to find business capabilities, propose bounded contexts, compare a business split with a technical-layer split, then write a three-line decision record.

DDD helps identify candidate boundaries; it does not require deploying every candidate as a separate service immediately. Review low coupling, high cohesion and context-specific models in the slides.

**At a glance**

![Lab 03 at a glance: Brief, then Business capabilities (3 labels), then gate, then Boundary options (2 or 3), then gate, then Trade-offs (+ / − table), then gate, then Decision record (Context · Decision · Consequence), then gate.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/03.svg)

## Review your microservices lectures

- **Microservices Lecture 2 - Condensed DDD**, slides 9, 10, 11, 17, 18, 21, 26, 32, 60, 70, 72, 73.

**Recall:** Event storming; Bounded contexts and ubiquitous language; Business capability vs technical-layer decomposition.

Before running the example:

1. Why can a technical-layer split create a distributed monolith?
2. Why is a bounded context a service candidate rather than an automatic deployment decision?

Slide numbers count from the title slide, including appendices. The small scenarios and code are teaching adaptations, not production services. Use the lecture decks supplied in class; they are not uploaded with this course.

## The idea in context

A fixed prompt chain exposes intermediate results. Gates can catch a malformed handoff while leaving domain judgement for review.

A bounded context is a model boundary; validate it before choosing deployment boundaries.

| Key term | Meaning |
| --- | --- |
| **Workflow** | A sequence of steps fixed by code; the model does not choose what happens next. |
| **Decomposition** | Splitting a large task into smaller steps with clear inputs and outputs. |
| **Gate** | A plain-code check between two steps that stops a bad result from spreading. |
| **Decision record (ADR)** | A short note of an architecture decision: its context, the decision, and its consequence. |

## Pattern · Single-Path Planning

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Passive Goal Creator.*

| | |
| --- | --- |
| **Context** | Complex goals need several dependent steps, where each result informs the next. |
| **Problem** | One monolithic response cannot reliably manage dependencies and accumulated intermediate results. |
| **Solution** | Make one ordered plan of a few steps, execute each step with the accumulated context, then synthesize the outputs. |
| **Use when** | The work is naturally sequential; steps depend on earlier outputs; one reasonable path is enough. |
| **Trade-off** | Easy to follow, but an early planning mistake propagates through the whole run. |
| **Flow** | Plan → Step 1 → Step N → Synthesize |
| **Quality attributes** | Benefits: Predictability, Performance, Observability, Maintainability. Liabilities: Reliability, Latency, Usability, Adaptability. |
| **In this lab** | Events lead to candidate bounded contexts, then a comparison of boundary strategies and a short decision record. Gates check output structure; domain experts still validate the proposed boundaries. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · A brief and a reusable gate
The gates check structural contracts. Students still judge whether a proposed context has coherent language, rules and data.

In [ ]:
import json
brief = ("ShopEasy sells jackets. A customer places an order; stock is reserved; payment is authorized; "
         "a parcel is shipped. Order, Inventory and Shipping currently share tables and releases.")
events = ["OrderPlaced", "StockReserved", "PaymentAuthorized", "ParcelShipped"]
def run_step(name, prompt, example, gate):
    reply = ask(prompt, example=example)
    print(name, reply)
    problem = gate(reply)
    if problem:
        raise ValueError(problem)
    return reply

def three_items(reply):
    items = json.loads(reply)
    return None if isinstance(items, list) and len(items) == 3 else "Expected a JSON list of three items."
print(events)

## Step 2 · From events to candidate bounded contexts
Event storming names facts in the past tense. Group by business meaning rather than by UI, logic and data-access layers. Our compact exercise groups Payment with a provided external capability; it does not claim all domains have exactly three contexts.

In [ ]:
prompt = (f"Brief: {brief}\nEvents: {events}\nPropose exactly 3 business bounded contexts for "
          "OrderPlaced, StockReserved and ParcelShipped. Payment is supplied externally. "
          "Reply with a JSON list of context labels only.")
contexts = json.loads(run_step("contexts", prompt, '["Order", "Inventory", "Shipping"]', three_items))
options = ["business capability split", "technical layer split"]
criteria = ["cohesive business rules", "independent change", "private data ownership"]
print("Compare:", options, "against", criteria)

## Step 3 · Compare the boundary choices
A technical-layer split can create a distributed monolith: one feature change spans several services. A business split needs stable contracts and private data to earn its independence.

In [ ]:
def every_cell(reply):
    table = json.loads(reply)
    if not isinstance(table, dict):
        return "Expected an object."
    missing = [f"{o}/{r}" for o in options for r in criteria if table.get(o, {}).get(r) not in ("+", "-")]
    return f"Missing/invalid marks: {missing}" if missing else None
prompt = (f"Brief: {brief}\nContexts: {contexts}\nCompare {options} using {criteria}. "
          'JSON only: {"<option>": {"<criterion>": "+" or "-"}}. Marks are qualitative, not measured.')
example = json.dumps({"business capability split": dict.fromkeys(criteria, "+"),
                      "technical layer split": dict.fromkeys(criteria, "-")})
tradeoffs = json.loads(run_step("trade-offs", prompt, example, every_cell))

## Step 4 · Record a decision and its cost
The ADR chooses a boundary strategy, not the number of deployed services. Shared identity can cross contracts without sharing the entire internal model.

In [ ]:
def record_gate(reply):
    fields = dict(line.split(": ", 1) for line in reply.strip().splitlines() if ": " in line)
    if list(fields) != ["Context", "Decision", "Consequence"]:
        return "Expected Context, Decision, Consequence."
    return None if fields["Decision"].strip().lower() in options else "Unknown boundary strategy."
prompt = (f"Brief: {brief}\nContexts: {contexts}\nTrade-offs: {json.dumps(tradeoffs)}\n"
          "Write exactly Context:, Decision:, Consequence: lines. Decision is one option label only. "
          "Consequence must name a cost; do not assume the service candidates are already deployed.")
example = ("Context: ShopEasy's shared tables and releases couple unrelated business changes.\n"
           "Decision: business capability split\n"
           "Consequence: Stable contracts and data ownership need design work; separate deployment can follow later.")
record = run_step("ADR", prompt, example, record_gate)

## What you should see
The chain proposes Order, Inventory and Shipping contexts, compares both strategies, and records the business-capability split with its contract/data-ownership cost. Three model calls make intermediate results visible. Gates verify shape, not the truth of every architectural claim.

## Try it
1. Replace Order, Inventory, Shipping with UI, BusinessLogic, DataAccess. The list gate still passes: explain why the design may still be poor.
2. Explain why `Product` has different fields in Catalog, Inventory and Shipping (DDD slides 72–73).
3. Add Returns and Refunds from DDD slide 60. Which language, rules and data would you investigate before drawing another boundary?

## Takeaway
A chain makes intermediate modelling choices reviewable. Good JSON is necessary for automation; domain experts and change patterns still validate service candidates.